# ASD Research: Multilayer Microbiome Feature Integration from Diagnosis to Behavioral Prediction
## Stage 1: Data Integration and Preprocessing

---

**Version**: V4.7 Revised Edition  
**Date**: February 2026  

### V4.7 Study Design

**Discovery Cohort**: Chinese multicenter (6 public cohorts) + Changchun cohort = **471 samples**  
**External Validation**: Moscow cohort = **74 samples** (cross-ethnic validation)

### Goals of This Stage

1. Load raw MetaPhlAn/HUMAnN outputs from each cohort
2. **Directly merge the discovery cohorts** (Chinese multicenter + Changchun)
3. Perform feature filtering (low prevalence, low abundance)
4. Extract group and batch information
5. Save preprocessed data for Stage 2

### Output Data (stage1_preprocessed_data.pkl)

```python
{
    'discovery_data_filtered': {...},  # Four-level discovery cohort data
    'moscow_data_filtered': {...},     # Four-level Moscow cohort data
    'discovery_group': Series,         # Discovery cohort groups (ASD/TD)
    'discovery_study': Series,         # Discovery cohort batches (7 StudyIDs)
    'moscow_group': Series,            # Moscow cohort groups
    'local_cohort_samples': list,      # Changchun sample ID list
    'local_group': Series,             # Changchun sub-cohort groups
    'metadata': dict                   # Metadata tables
}
```


## 1.1 Environment Setup

In [ ]:
# ============================================================
# Install required dependencies
# ============================================================
!pip install pandas numpy scipy scikit-learn matplotlib seaborn -q

print("✓ 依赖包安装完成")

✓ 依赖包安装完成


In [ ]:
# ============================================================
# Import required libraries
# ============================================================
import os
import glob
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ 库导入完成")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

✓ 库导入完成
  pandas: 2.2.2
  numpy: 2.0.2


In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✓ Google Drive 已挂载")
else:
    print("✓ Google Drive 已经挂载")

Mounted at /content/drive
✓ Google Drive 已挂载


In [ ]:
# ============================================================
# Define project paths
# ============================================================
BASE_PATH = '/content/drive/MyDrive/ASD_Research'
RAW_PATH = os.path.join(BASE_PATH, '01_raw_outputs')
MERGED_PATH = os.path.join(BASE_PATH, '02_merged_data')
META_PATH = os.path.join(BASE_PATH, '03_metadata_tables')
FIG_PATH = os.path.join(BASE_PATH, '04_figures')

# Create required directories
for path in [MERGED_PATH, FIG_PATH]:
    os.makedirs(path, exist_ok=True)

print("✓ 路径设置完成")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  RAW_PATH: {RAW_PATH}")

✓ 路径设置完成
  BASE_PATH: /content/drive/MyDrive/ASD_Research
  RAW_PATH: /content/drive/MyDrive/ASD_Research/01_raw_outputs


## 1.2 Define Cohort Groups

In [ ]:
# ============================================================
# Define cohort groups (V4.7 revision)
# ============================================================

# Six public Chinese cohorts
CHINA_PUBLIC_COHORTS = [
    'Study_Dan2020',      # 香港
    'Study_Wang2019',     # 大陆
    'Study_Xu2023',       # 大陆
    'Study_Zhang2020',    # 大陆
    'Study_Tong2022',     # 大陆
    'Study_CUHK'          # 香港
]

# Changchun in-house cohort (included in the discovery cohort)
LOCAL_COHORT = 'Local_Cohort'

# Discovery cohort = public cohorts + Changchun cohort (7 total)
DISCOVERY_COHORTS = CHINA_PUBLIC_COHORTS + [LOCAL_COHORT]

# External validation cohort (Moscow, cross-ethnic validation)
MOSCOW_COHORT = 'Study_Kovtun2020'

# Data types and their file patterns (dictionary format)
DATA_TYPES = {
    'taxa': {
        'subdir': 'taxa',
        'patterns': ['*_taxonomic_profile.tsv', '*_metaphlan.tsv', '*_profile.tsv', '*.tsv']
    },
    'pathways': {
        'subdir': 'pathways',
        'patterns': ['*_pathabundance_relab.tsv', '*_pathabundance.tsv', '*.tsv']
    },
    'genes': {
        'subdir': 'genes',
        'patterns': ['*_genefamilies_relab.tsv', '*_genefamilies.tsv', '*.tsv']
    },
    'ecs': {
        'subdir': 'ecs',
        'patterns': ['*_ecs_relab.tsv', '*_ecs.tsv', '*.tsv']
    }
}

# Data type list (used for iteration)
DATA_TYPE_LIST = ['taxa', 'pathways', 'genes', 'ecs']

print("【V4.7队列设计】")
print(f"  发现队列: {len(DISCOVERY_COHORTS)}个中国队列")
for c in DISCOVERY_COHORTS:
    print(f"    - {c}")
print(f"  外部验证: {MOSCOW_COHORT} (跨种族)")

【V4.7队列设计】
  发现队列: 7个中国队列
    - Study_Dan2020
    - Study_Wang2019
    - Study_Xu2023
    - Study_Zhang2020
    - Study_Tong2022
    - Study_CUHK
    - Local_Cohort
  外部验证: Study_Kovtun2020 (跨种族)


## 1.3 Define Data Loading Functions

In [ ]:
# ============================================================
# Define a function to read MetaPhlAn species abundance files
# ============================================================
def read_metaphlan_file(filepath):
    """Read species abundance files generated by MetaPhlAn."""
    try:
        # Try different reading strategies
        with open(filepath, 'r') as f:
            first_lines = [f.readline() for _ in range(10)]

        # Find the header row
        skip_rows = 0
        for i, line in enumerate(first_lines):
            if line.startswith('#') and not line.startswith('#clade'):
                skip_rows = i + 1
            elif line.startswith('clade_name') or line.startswith('#clade'):
                skip_rows = i
                break

        df = pd.read_csv(filepath, sep='\t', skiprows=skip_rows)

        # Standardize column names
        if 'clade_name' in df.columns or '#clade_name' in df.columns:
            name_col = 'clade_name' if 'clade_name' in df.columns else '#clade_name'
            abund_col = 'relative_abundance' if 'relative_abundance' in df.columns else df.columns[1]

            result = df[[name_col, abund_col]].copy()
            result.columns = ['feature', 'abundance']
            return result
        else:
            # Assume the first column is the feature name and the second column is abundance
            result = df.iloc[:, :2].copy()
            result.columns = ['feature', 'abundance']
            return result

    except Exception as e:
        print(f"    读取失败: {str(e)[:50]}")
        return None

print("✓ MetaPhlAn读取函数定义完成")

✓ MetaPhlAn读取函数定义完成


In [ ]:
# ============================================================
# Define a function to read HUMAnN functional abundance files
# ============================================================
def read_humann_file(filepath):
    """Read functional abundance files generated by HUMAnN."""
    try:
        df = pd.read_csv(filepath, sep='\t')

        # The first column contains feature names
        name_col = df.columns[0]
        abund_col = df.columns[1] if len(df.columns) > 1 else None

        if abund_col is None:
            return None

        result = df[[name_col, abund_col]].copy()
        result.columns = ['feature', 'abundance']

        # Remove species stratification information (the part after '|')
        result['feature'] = result['feature'].apply(lambda x: x.split('|')[0] if '|' in str(x) else x)

        # Aggregate identical features
        result = result.groupby('feature')['abundance'].sum().reset_index()

        return result

    except Exception as e:
        print(f"    读取失败: {str(e)[:50]}")
        return None

print("✓ HUMAnN读取函数定义完成")

✓ HUMAnN读取函数定义完成


In [ ]:
# ============================================================
# Define a function to extract sample IDs from filenames
# ============================================================
def extract_sample_id(filename):
    """Extract the sample ID from a filename."""
    # Remove the file extension
    base = os.path.splitext(filename)[0]

    # Remove common suffixes
    suffixes = ['_taxonomic_profile', '_metaphlan', '_profile',
                '_pathabundance_relab', '_pathabundance',
                '_genefamilies_relab', '_genefamilies',
                '_ecs_relab', '_ecs']

    for suffix in suffixes:
        if base.endswith(suffix):
            base = base[:-len(suffix)]
            break

    return base

print("✓ 样本ID提取函数定义完成")

✓ 样本ID提取函数定义完成


In [ ]:
# ============================================================
# Define a function to merge data from a single cohort
# ============================================================
def merge_cohort_data(cohort_name, data_type):
    """Merge all sample data from a single cohort."""
    cohort_path = os.path.join(RAW_PATH, cohort_name)

    if not os.path.exists(cohort_path):
        print(f"  ⚠ 队列目录不存在: {cohort_name}")
        return None, {}

    # Get the data subdirectory
    data_info = DATA_TYPES[data_type]
    data_dir = os.path.join(cohort_path, data_info['subdir'])

    if not os.path.exists(data_dir):
        print(f"  ⚠ 数据目录不存在: {data_dir}")
        return None, {}

    # Find data files
    files = []
    for pattern in data_info['patterns']:
        found = glob.glob(os.path.join(data_dir, pattern))
        if found:
            files = found
            break

    if not files:
        print(f"  ⚠ 未找到数据文件: {cohort_name}/{data_type}")
        return None, {}

    # Read and merge
    all_data = {}
    sample_cohort_map = {}

    for filepath in files:
        filename = os.path.basename(filepath)
        sample_id = extract_sample_id(filename)

        # Select the loading function based on the data type
        if data_type == 'taxa':
            df = read_metaphlan_file(filepath)
        else:
            df = read_humann_file(filepath)

        if df is not None and len(df) > 0:
            all_data[sample_id] = df.set_index('feature')['abundance']
            sample_cohort_map[sample_id] = cohort_name

    if not all_data:
        return None, {}

    # Merge into a DataFrame
    merged_df = pd.DataFrame(all_data)
    merged_df = merged_df.fillna(0)

    print(f"  ✓ {cohort_name}: {merged_df.shape[1]} 样本, {merged_df.shape[0]} 特征")

    return merged_df, sample_cohort_map

print("✓ 队列合并函数定义完成")

✓ 队列合并函数定义完成


In [ ]:
# ============================================================
# Define a function to merge data from multiple cohorts
# ============================================================
def merge_multiple_cohorts(cohort_list, data_type):
    """Merge data from multiple cohorts."""
    all_dfs = []
    all_sample_map = {}

    for cohort in cohort_list:
        df, sample_map = merge_cohort_data(cohort, data_type)
        if df is not None:
            all_dfs.append(df)
            all_sample_map.update(sample_map)

    if not all_dfs:
        return None, {}

    # Merge all cohorts and keep shared features
    # Use an outer join to keep all features and fill missing values with 0
    merged = pd.concat(all_dfs, axis=1, join='outer')
    merged = merged.fillna(0)

    return merged, all_sample_map

print("✓ 多队列合并函数定义完成")

✓ 多队列合并函数定义完成


## 1.4 Run Data Merging

In [ ]:
# ============================================================
# Merge discovery cohort data (7 Chinese cohorts)
# ============================================================
print("=" * 60)
print("合并发现队列数据（V4.7: 7个中国队列）")
print("=" * 60)

discovery_data = {}
discovery_sample_cohort_map = {}

for dtype in DATA_TYPE_LIST:
    print(f"\n--- 处理 {dtype} 数据 ---")
    df, sample_map = merge_multiple_cohorts(DISCOVERY_COHORTS, dtype)

    if df is not None:
        discovery_data[dtype] = df
        discovery_sample_cohort_map.update(sample_map)
        print(f"  合并完成: {df.shape[0]} 特征 × {df.shape[1]} 样本")
    else:
        discovery_data[dtype] = None
        print(f"  ⚠️ 合并失败")

print(f"\n【发现队列汇总】")
print(f"  总样本数: {len(discovery_sample_cohort_map)}")
for dtype, df in discovery_data.items():
    if df is not None:
        print(f"  {dtype}: {df.shape[1]} 样本, {df.shape[0]} 特征")

合并发现队列数据（V4.7: 7个中国队列）

--- 处理 taxa 数据 ---
  ✓ Study_Dan2020: 60 样本, 2406 特征
  ✓ Study_Wang2019: 61 样本, 2280 特征
  ✓ Study_Xu2023: 21 样本, 1529 特征
  ✓ Study_Zhang2020: 79 样本, 2552 特征
  ✓ Study_Tong2022: 52 样本, 2391 特征
  ✓ Study_CUHK: 128 样本, 3202 特征
  ✓ Local_Cohort: 70 样本, 2829 特征
  合并完成: 4508 特征 × 471 样本

--- 处理 pathways 数据 ---
  ✓ Study_Dan2020: 60 样本, 502 特征
  ✓ Study_Wang2019: 61 样本, 486 特征
  ✓ Study_Xu2023: 21 样本, 450 特征
  ✓ Study_Zhang2020: 79 样本, 497 特征
  ✓ Study_Tong2022: 52 样本, 479 特征
  ✓ Study_CUHK: 128 样本, 532 特征
  ✓ Local_Cohort: 70 样本, 555 特征
  合并完成: 585 特征 × 471 样本

--- 处理 genes 数据 ---
  ✓ Study_Dan2020: 60 样本, 1886671 特征
  ✓ Study_Wang2019: 61 样本, 1875186 特征
  ✓ Study_Xu2023: 21 样本, 1151859 特征
  ✓ Study_Zhang2020: 79 样本, 1912983 特征
  ✓ Study_Tong2022: 52 样本, 1778290 特征
  ✓ Study_CUHK: 128 样本, 2680364 特征
  ✓ Local_Cohort: 70 样本, 2227935 特征
  合并完成: 3488516 特征 × 471 样本

--- 处理 ecs 数据 ---
  ✓ Study_Dan2020: 60 样本, 2339 特征
  ✓ Study_Wang2019: 61 样本, 2209 特征
  ✓ Study_Xu2023: 2

In [ ]:
# ============================================================
# Merge Moscow cohort data (external validation)
# ============================================================
print("=" * 60)
print("合并莫斯科队列数据（跨种族外部验证）")
print("=" * 60)

moscow_data = {}
moscow_sample_cohort_map = {}

for dtype in DATA_TYPE_LIST:
    print(f"\n--- 处理 {dtype} 数据 ---")
    df, sample_map = merge_cohort_data(MOSCOW_COHORT, dtype)

    if df is not None:
        moscow_data[dtype] = df
        moscow_sample_cohort_map.update(sample_map)
    else:
        moscow_data[dtype] = None

print(f"\n【莫斯科队列汇总】")
print(f"  总样本数: {len(moscow_sample_cohort_map)}")
for dtype, df in moscow_data.items():
    if df is not None:
        print(f"  {dtype}: {df.shape[1]} 样本, {df.shape[0]} 特征")

合并莫斯科队列数据（跨种族外部验证）

--- 处理 taxa 数据 ---
  ✓ Study_Kovtun2020: 74 样本, 2842 特征

--- 处理 pathways 数据 ---
  ✓ Study_Kovtun2020: 74 样本, 489 特征

--- 处理 genes 数据 ---
  ✓ Study_Kovtun2020: 74 样本, 2008014 特征

--- 处理 ecs 数据 ---
  ✓ Study_Kovtun2020: 74 样本, 2254 特征

【莫斯科队列汇总】
  总样本数: 74
  taxa: 74 样本, 2842 特征
  pathways: 74 样本, 489 特征
  genes: 74 样本, 2008014 特征
  ecs: 74 样本, 2254 特征


## 1.5 Load Metadata

In [ ]:
# ============================================================
# 1.5 Load metadata (V4.7 final fixed version)
# ============================================================
import pandas as pd
import os

print("=" * 60)
print("加载元数据表 (V4.7 最终修复版)")
print("=" * 60)

metadata = {}
analytical_meta = None
sample_id_col = None
group_col = None
study_col = None

# 1. Define file paths
meta_files = {
    'analytical_metadata': 'Table3_Analytical_Metadata.csv',
    'sra_metadata': 'Table2_SRA_Metadata_Raw.csv'
}

for key, filename in meta_files.items():
    filepath = os.path.join(META_PATH, filename)
    if os.path.exists(filepath):
        try:
            df = pd.read_csv(filepath)
            metadata[key] = df
            print(f"  ✓ 成功读取 {filename}: {df.shape[0]} 行")
            # Print column names for debugging (if this still fails, we can inspect the actual column names)
            print(f"    列名预览: {df.columns.tolist()[:10]}")

            if key == 'analytical_metadata':
                analytical_meta = df
        except Exception as e:
            print(f"  ⚠ 读取 {filename} 失败: {e}")

# 2. Intelligently detect column names (with added GroupID support)
if analytical_meta is not None:
    cols = analytical_meta.columns.tolist()

    # --- Detect SampleID ---
    potential_sample_cols = ['Sample_ID', 'SampleID', 'Run', 'run', 'Run_s', 'run_accession', 'sample_name', 'sample_id', 'Accession']
    for candidate in potential_sample_cols:
        if candidate in cols:
            sample_id_col = candidate
            print(f"    -> 识别到样本ID列: '{sample_id_col}'")
            break

    # --- Detect Group (key update: added GroupID support) ---
    potential_group_cols = [
        'GroupID', 'Group_ID', 'Group', 'group',  # <--- 最可能的列名放在前面
        'Diagnosis', 'diagnosis', 'disease_status', 'status', 'Status',
        'Assay_Type', 'Condition', 'condition', 'Class', 'class'
    ]
    for candidate in potential_group_cols:
        if candidate in cols:
            group_col = candidate
            print(f"    -> 识别到分组列: '{group_col}'")
            break

    # --- Detect StudyID ---
    potential_study_cols = ['Study_ID', 'StudyID', 'BioProject', 'bioproject', 'Project', 'project', 'study_id', 'Study', 'study', 'Cohort', 'cohort']
    for candidate in potential_study_cols:
        if candidate in cols:
            study_col = candidate
            print(f"    -> 识别到队列列: '{study_col}'")
            break

加载元数据表 (V4.7 最终修复版)
  ✓ 成功读取 Table3_Analytical_Metadata.csv: 545 行
    列名预览: ['Sample_ID', 'Study_ID', 'Batch', 'Condition', 'Region', 'Age_Category', 'Sex']
  ✓ 成功读取 Table2_SRA_Metadata_Raw.csv: 578 行
    列名预览: ['Run_Accession', 'Study_ID', 'Base_Count_Gb', 'Read_Count_M', 'Instrument_Model', 'Library_Layout', 'Library_Strategy', 'QC_Status', 'Exclusion_Reason']
    -> 识别到样本ID列: 'Sample_ID'
    -> 识别到分组列: 'Condition'
    -> 识别到队列列: 'Study_ID'


In [ ]:
# ============================================================
# 1.6 Extract Group Information
# ============================================================
print("\n" + "=" * 60)
print("提取分组信息")
print("=" * 60)

# Initialize
discovery_group = pd.Series(dtype=str)
discovery_study = pd.Series(dtype=str)
moscow_group = pd.Series(dtype=str)
local_group = pd.Series(dtype=str)

if analytical_meta is not None and sample_id_col and group_col:
    # 1. Standardize group labels
    analytical_meta[group_col] = analytical_meta[group_col].replace({
        'Autism': 'ASD', 'autism': 'ASD', 'ASD': 'ASD', 'asd': 'ASD',
        'Control': 'TD', 'control': 'TD', 'Healthy': 'TD', 'TD': 'TD',
        'Typically Developing': 'TD'
    })

    # 2. Extract discovery cohort groups (6 public cohorts)
    public_samples = [s for s, c in discovery_sample_cohort_map.items() if c != 'Local_Cohort']
    matched_meta = analytical_meta[analytical_meta[sample_id_col].isin(public_samples)]
    discovery_group = matched_meta.set_index(sample_id_col)[group_col]

    print(f"  ✓ 公开队列匹配成功: {len(discovery_group)} 样本")

    # 3. Process the Changchun cohort (Local_Cohort) with automatic filename inference
    local_samples = [s for s, c in discovery_sample_cohort_map.items() if c == 'Local_Cohort']
    if len(local_samples) > 0:
        local_in_meta = analytical_meta[analytical_meta[sample_id_col].isin(local_samples)]

        if len(local_in_meta) > 0:
            local_group = local_in_meta.set_index(sample_id_col)[group_col]
        else:
            print("  ⚠ 表中未找到长春样本，启用【文件名推断模式】")
            inferred_groups = {}
            for s in local_samples:
                # Simple rule: if the filename contains ASD, label it as ASD; otherwise, label it as TD
                if 'ASD' in s.upper():
                    inferred_groups[s] = 'ASD'
                else:
                    inferred_groups[s] = 'TD'
            local_group = pd.Series(inferred_groups)

        # Add Changchun groups into the discovery cohort
        new_local = local_group[~local_group.index.isin(discovery_group.index)]
        discovery_group = pd.concat([discovery_group, new_local])

    # 4. Build the StudyID series
    discovery_study = pd.Series(discovery_sample_cohort_map)
    discovery_study = discovery_study[discovery_study.index.isin(discovery_group.index)]
    discovery_group = discovery_group.loc[discovery_study.index] # 对齐

    # 5. Extract Moscow cohort groups
    moscow_samples = list(moscow_sample_cohort_map.keys())
    moscow_in_meta = analytical_meta[analytical_meta[sample_id_col].isin(moscow_samples)]
    moscow_group = moscow_in_meta.set_index(sample_id_col)[group_col]

    print(f"\n【最终分组统计】")
    print(f"  发现队列 (n={len(discovery_group)}): {discovery_group.value_counts().to_dict()}")
    print(f"  莫斯科队列 (n={len(moscow_group)}): {moscow_group.value_counts().to_dict()}")

else:
    print("❌ 依然无法识别关键列！")
    print(f"  当前识别状态 -> SampleID: {sample_id_col}, Group: {group_col}, StudyID: {study_col}")
    if analytical_meta is not None:
        print(f"  CSV文件实际列名: {analytical_meta.columns.tolist()}")


提取分组信息
  ✓ 公开队列匹配成功: 401 样本

【最终分组统计】
  发现队列 (n=471): {'ASD': 245, 'TD': 226}
  莫斯科队列 (n=74): {'ASD': 56, 'TD': 18}


## 1.6 Data Quality Control and Feature Filtering

In [ ]:
# ============================================================
# Define the feature filtering function (V4.7 rigorous version)
# ============================================================
def filter_features_rigorous(df, group_series,
                             prevalence_threshold=0.05,
                             abundance_threshold=1e-5,
                             detection_limit=1e-7):
    """
    严谨的特征过滤函数：支持分组敏感性过滤

    核心逻辑：
    保留满足以下任一条件的特征：
    1. (ASD组患病率 >= 阈值) OR (TD组患病率 >= 阈值)
       AND
    2. 平均相对丰度 >= 阈值 (可选，有些研究只看患病率)

    Parameters:
    -----------
    df : pd.DataFrame
        特征矩阵（行为特征，列为样本）
    group_series : pd.Series
        样本分组信息（索引为样本ID，值为分组标签）
    prevalence_threshold : float
        最小患病率阈值（默认 0.05, 即 5%）
    abundance_threshold : float
        最小平均丰度阈值（默认 1e-5）
    detection_limit : float
        判定特征存在的最小丰度值（默认 1e-7，排除噪音）
    """
    if df is None or group_series is None:
        print("⚠ 输入数据为空")
        return None

    # 1. Ensure sample alignment
    common_samples = df.columns.intersection(group_series.index)
    if len(common_samples) < len(df.columns):
        print(f"⚠ 警告: 仅 {len(common_samples)}/{len(df.columns)} 个样本有分组信息，将仅使用这些样本进行过滤判定")

    df_aligned = df[common_samples]
    groups = group_series.loc[common_samples]

    original_features = df.shape[0]

    # 2. Compute the mean abundance (Global Mean Abundance)
    # Only features with extremely low mean abundance are removed globally
    mean_abundance = df_aligned.mean(axis=1)
    mask_abundance = mean_abundance >= abundance_threshold

    # 3. Compute group-wise prevalence
    # Presence criterion: abundance > detection_limit (more rigorous than simply > 0)
    present_matrix = (df_aligned > detection_limit)

    # Compute prevalence for each group separately
    groups_list = groups.unique()
    prevalence_mask = pd.Series(False, index=df.index)

    print("  分组患病率检查:")
    for g in groups_list:
        # Get samples from the current group
        group_samples = groups[groups == g].index
        # Compute prevalence in the current group
        group_prev = present_matrix[group_samples].sum(axis=1) / len(group_samples)
        # Update the mask: keep the feature if it meets the threshold in any group (logical OR)
        prevalence_mask = prevalence_mask | (group_prev >= prevalence_threshold)

        n_pass = (group_prev >= prevalence_threshold).sum()
        print(f"    - {g}组 (n={len(group_samples)}): {n_pass} 个特征达标")

    # 4. Combined filtering (logical AND)
    # Keep features that are (prevalent in any group) AND (meet the global mean abundance threshold)
    final_keep_mask = prevalence_mask & mask_abundance

    df_filtered = df.loc[final_keep_mask]

    # 5. Output the report
    filtered_features = df_filtered.shape[0]
    removed_count = original_features - filtered_features

    print("-" * 40)
    print(f"特征过滤结果:")
    print(f"  原始特征: {original_features}")
    print(f"  保留特征: {filtered_features}")
    print(f"  移除特征: {removed_count} ({(removed_count/original_features)*100:.1f}%)")
    print(f"  阈值设置: Prev>={prevalence_threshold:.0%}, Abund>={abundance_threshold}, Detect>{detection_limit}")
    print("-" * 40)

    return df_filtered

print("✓ 严谨版特征过滤函数定义完成")

✓ 严谨版特征过滤函数定义完成


In [ ]:
# ============================================================
# Run feature filtering (V4.7 rigorous version, group-sensitive)
# ============================================================
print("=" * 60)
print("执行特征过滤 (保留分组特异性标志物)")
print("=" * 60)

# 1. Discovery cohort filtering
# Pass in discovery_group to ensure that features enriched only in the ASD or TD group are retained
print("\n--- 发现队列 (中国多中心 + 长春) ---")
discovery_data_filtered = {}
for dtype, df in discovery_data.items():
    print(f"\n[处理 {dtype} 数据]:")
    if df is not None:
        discovery_data_filtered[dtype] = filter_features_rigorous(
            df,
            discovery_group,          # <--- 关键修改：传入分组信息
            prevalence_threshold=0.05, # 保持 5% 阈值
            abundance_threshold=1e-5   # 保持 1e-5 丰度阈值
        )
    else:
        discovery_data_filtered[dtype] = None

# 2. Moscow cohort filtering
# Also pass in moscow_group to remove cohort-specific noise
print("\n--- 莫斯科队列 (跨种族验证) ---")
moscow_data_filtered = {}
for dtype, df in moscow_data.items():
    print(f"\n[处理 {dtype} 数据]:")
    if df is not None:
        moscow_data_filtered[dtype] = filter_features_rigorous(
            df,
            moscow_group,             # <--- 关键修改：传入分组信息
            prevalence_threshold=0.05,
            abundance_threshold=1e-5
        )
    else:
        moscow_data_filtered[dtype] = None

print("\n" + "="*60)
print("✓ 特征过滤完成")
print("  提示：Stage 5 建模时将取两者的'特征交集'，")
print("  但在此阶段，我们独立过滤以确保各队列数据的纯净性。")

执行特征过滤 (保留分组特异性标志物)

--- 发现队列 (中国多中心 + 长春) ---

[处理 taxa 数据]:
  分组患病率检查:
    - ASD组 (n=245): 1545 个特征达标
    - TD组 (n=226): 1519 个特征达标
----------------------------------------
特征过滤结果:
  原始特征: 4508
  保留特征: 1658
  移除特征: 2850 (63.2%)
  阈值设置: Prev>=5%, Abund>=1e-05, Detect>1e-07
----------------------------------------

[处理 pathways 数据]:
  分组患病率检查:
    - ASD组 (n=245): 470 个特征达标
    - TD组 (n=226): 466 个特征达标
----------------------------------------
特征过滤结果:
  原始特征: 585
  保留特征: 433
  移除特征: 152 (26.0%)
  阈值设置: Prev>=5%, Abund>=1e-05, Detect>1e-07
----------------------------------------

[处理 genes 数据]:
  分组患病率检查:
    - ASD组 (n=245): 1125922 个特征达标
    - TD组 (n=226): 1099953 个特征达标
----------------------------------------
特征过滤结果:
  原始特征: 3488516
  保留特征: 37334
  移除特征: 3451182 (98.9%)
  阈值设置: Prev>=5%, Abund>=1e-05, Detect>1e-07
----------------------------------------

[处理 ecs 数据]:
  分组患病率检查:
    - ASD组 (n=245): 2112 个特征达标
    - TD组 (n=226): 2107 个特征达标
----------------------------------------
特征过滤结果

## 1.7 Save Data

In [ ]:
# ============================================================
# Save Stage 1 output data
# ============================================================
print("=" * 60)
print("保存Stage1输出数据")
print("=" * 60)

stage1_output = {
    # ==== Filtered data ====
    'discovery_data_filtered': discovery_data_filtered,  # 发现队列
    'moscow_data_filtered': moscow_data_filtered,        # 莫斯科队列

    # ==== Group information (critical) ====
    'discovery_group': discovery_group,      # 发现队列分组 (ASD/TD)
    'discovery_study': discovery_study,      # 发现队列批次 (7个StudyID)
    'moscow_group': moscow_group,            # 莫斯科队列分组

    # ==== Changchun cohort labels (used in Stage 6) ====
    'local_cohort_samples': local_cohort_samples,  # 长春样本ID列表
    'local_group': local_group,                     # 长春子队列分组

    # ==== Sample-to-cohort mapping ====
    'discovery_sample_cohort_map': discovery_sample_cohort_map,
    'moscow_sample_cohort_map': moscow_sample_cohort_map,

    # ==== Metadata ====
    'metadata': metadata,
    'column_names': {
        'sample_id_col': sample_id_col,
        'group_col': group_col,
        'study_col': study_col
    },

    # ==== Version information ====
    'version': 'V4.7',
    'description': '发现队列(7个中国队列) + 外部验证(莫斯科)'
}

# Save
output_path = os.path.join(MERGED_PATH, 'stage1_preprocessed_data.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(stage1_output, f)

print(f"✓ 数据已保存至: {output_path}")

# Validate saved contents
print("\n【输出数据结构验证】")
for key in ['discovery_data_filtered', 'moscow_data_filtered',
            'discovery_group', 'discovery_study', 'moscow_group',
            'local_cohort_samples', 'local_group']:
    value = stage1_output.get(key)
    if isinstance(value, dict):
        print(f"  ✓ {key}: dict with {len(value)} items")
    elif isinstance(value, pd.Series):
        print(f"  ✓ {key}: Series with {len(value)} items")
    elif isinstance(value, list):
        print(f"  ✓ {key}: list with {len(value)} items")
    elif value is None:
        print(f"  ⚠ {key}: None")
    else:
        print(f"  ✓ {key}: {type(value).__name__}")

保存Stage1输出数据
✓ 数据已保存至: /content/drive/MyDrive/ASD_Research/02_merged_data/stage1_preprocessed_data.pkl

【输出数据结构验证】
  ✓ discovery_data_filtered: dict with 4 items
  ✓ moscow_data_filtered: dict with 4 items
  ✓ discovery_group: Series with 471 items
  ✓ discovery_study: Series with 471 items
  ✓ moscow_group: Series with 74 items
  ✓ local_cohort_samples: list with 70 items
  ✓ local_group: Series with 70 items


In [ ]:
# ============================================================
# Stage 1 summary report
# ============================================================
print("\n" + "=" * 60)
print("Stage1：数据整合与预处理 - 完成！")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────────────┐
│                    Stage1 处理汇总 (V4.7)                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  【发现队列】7个中国队列（直接合并）                             │
│  • 6个公开队列 + 长春队列                                       │
│  • 用途: Stage2-5 (批次校正、分析、建模)                        │
│                                                                 │
│  【外部验证队列】莫斯科队列                                      │
│  • 用途: Stage5 跨种族外部验证                                  │
│                                                                 │
│  【长春子队列标识】已保存                                        │
│  • 用途: Stage6 行为预测分析                                    │
│                                                                 │
│  【输出变量】供Stage2使用                                        │
│  • discovery_data_filtered: 发现队列4层面数据                   │
│  • discovery_group: 分组信息                                    │
│  • discovery_study: 批次信息（7个StudyID）                     │
│  • moscow_data_filtered: 莫斯科队列数据                         │
│  • moscow_group: 莫斯科分组                                     │
│  • local_cohort_samples: 长春样本ID                            │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
""")

print("\n【数据统计】")
print(f"  发现队列样本数: {len(discovery_sample_cohort_map)}")
print(f"  莫斯科队列样本数: {len(moscow_sample_cohort_map)}")
print(f"  长春子队列样本数: {len(local_cohort_samples)}")

print("\n" + "=" * 60)
print("✓ Stage1完成，请继续执行Stage2")
print("=" * 60)


Stage1：数据整合与预处理 - 完成！

┌─────────────────────────────────────────────────────────────────┐
│                    Stage1 处理汇总 (V4.7)                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  【发现队列】7个中国队列（直接合并）                             │
│  • 6个公开队列 + 长春队列                                       │
│  • 用途: Stage2-5 (批次校正、分析、建模)                        │
│                                                                 │
│  【外部验证队列】莫斯科队列                                      │
│  • 用途: Stage5 跨种族外部验证                                  │
│                                                                 │
│  【长春子队列标识】已保存                                        │
│  • 用途: Stage6 行为预测分析                                    │
│                                                                 │
│  【输出变量】供Stage2使用                                        │
│  • discovery_data_filtered: 发现队列4层面数据     